# Phase II - Step 5b: Real Jev (TypeSafe System One) as the decision model

Notebook 04 used a *surrogate* JEV. Here the alert decision is made by the real **Jev** model through the TypeSafe API
(<https://docs.typesafe.ai>), and compared with the surrogate and with the RF baseline using the same costs, thresholds protocol and bootstrap.

| question | type | use |
|---|---|---|
| `is_malicious` | Noul | P(attack) from the flow evidence; thresholded in code (validation-tuned) |
| `action` | Choice: `block` / `allow` / `escalate` | Jev's own typed decision + `confidence`; low confidence -> human (docs: confidence-gated routing) |
| `explanation_supports` | Score (0-2) | does the SHAP evidence make behavioural sense for the verdict |

**Two state variants** (to isolate what Jev adds):
- `flow_only` - flow features only (no RF, no XAI-Q). Jev alone.
- `full` - flow + RF probability + top SHAP features + XAI-Q (ESS/SI/AR). Jev as the decision layer on top of the pipeline.

**Never in the state:** `label` and `attack_cat` (that would leak the answer).

**Safety and cost controls**
- API key is read only from the environment variable `TYPESAFE_API_KEY` (never written to the notebook or outputs).
- `DRY_RUN = True` sends only a small stratified sample so you can check quality and cost first. Set `DRY_RUN = False` for the full run.
- Every answer is appended to `jev_api_cache.jsonl`; re-running never re-sends a request that is already cached.
- `JEV_MOCK=1` replaces the API with a local fake (pipeline test only; writes to a separate cache and must not be reported).

The model name/version returned by the API is recorded; report it in the paper (the docs note that values can change between Jev releases).

## 0. Setup

In [22]:
import importlib.util
import json
import os
import subprocess
import sys
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

load_dotenv(override=True)   # .env wins over a stale TYPESAFE_API_KEY already in the kernel/shell

warnings.filterwarnings('ignore')
SEED = 42
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

USE_MOCK = os.getenv('JEV_MOCK') == '1'
if not USE_MOCK and importlib.util.find_spec('typesafe_sdk') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'typesafe-sdk'])

# Static path: WSL Ubuntu view of the D: drive (falls back to the Windows path outside WSL)
ROOT = Path('/mnt/d/ai_security/xai-q-audit-ids') if Path('/mnt/d').exists() else Path(r'D:\ai_security\xai-q-audit-ids')
OUT_DIR = ROOT / 'data' / 'phaseII_outputs'
CACHE = OUT_DIR / ('jev_api_cache_MOCK.jsonl' if USE_MOCK else 'jev_api_cache.jsonl')

JEV_MODEL = 'jev-latest'
VARIANTS = ['flow_only', 'full']
DRY_RUN = False            # True: ~DRY_PER_CAT alerts per attack category per split. False: all sampled alerts
DRY_PER_CAT = 3
MAX_WORKERS = 4
MAX_RETRIES = 4
CONF_FLOOR = 0.5          # docs: confidence below ~0.5 -> do not act, route to a human
C_FN, C_FP, C_ESC = 20.0, 1.0, 0.3   # same cost model as notebook 04
N_BOOT = 1000

if not USE_MOCK and not os.getenv('TYPESAFE_API_KEY'):
    raise SystemExit('Set TYPESAFE_API_KEY in the environment of this kernel first (or JEV_MOCK=1 for a pipeline test).')
print('mock' if USE_MOCK else 'REAL API', '| model', JEV_MODEL, '| dry run:', DRY_RUN, '| cache:', CACHE.name)

REAL API | model jev-latest | dry run: False | cache: jev_api_cache.jsonl


## 1. Load signals and rebuild the raw (human-readable) flow records
`xaiq_*.parquet` index = row position in the saved split. Raw values come from `train.csv`/`test.csv` (same dedupe and ordering as notebook 02), and are verified against the saved labels.

In [14]:
DATA = ROOT / 'data' / 'unsw-nb15_mouwiya'
train_raw = pd.read_csv(DATA / 'train.csv')
test_raw = pd.read_csv(DATA / 'test.csv')
FEATURE_SRC = [c for c in train_raw.columns if c not in ('id', 'attack_cat', 'label')]
train_df = train_raw.drop_duplicates(FEATURE_SRC).reset_index(drop=True)

sig = {'val': pd.read_parquet(OUT_DIR / 'xaiq_val.parquet'), 'test': pd.read_parquet(OUT_DIR / 'xaiq_test.parquet')}
raw = {'val': train_df.loc[sig['val'].index], 'test': test_raw.loc[sig['test'].index]}
for s in sig:
    assert (raw[s]['label'].values == sig[s]['label'].values).all(), f'{s}: raw/label mismatch'
    assert (raw[s]['attack_cat'].values == sig[s]['attack_cat'].values).all(), f'{s}: raw/attack_cat mismatch'
    print(s, 'alerts:', len(sig[s]), '| raw/label alignment OK')

FLOW_COLS = [c for c in ['proto', 'service', 'state', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl',
                         'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'tcprtt', 'smean', 'dmean',
                         'ct_srv_src', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_state_ttl',
                         'ct_flw_http_mthd', 'is_ftp_login'] if c in train_raw.columns]


def clean(v):
    if isinstance(v, (np.floating, float)):
        return float(f'{float(v):.4g}')
    if isinstance(v, np.integer):
        return int(v)
    return v


def parse_top(s):
    out = []
    for tok in s.split(';'):
        f, v = tok.rsplit(':', 1)
        out.append({'feature': f, 'shap_toward_attack': float(v)})
    return out


def build_state(split, rid, variant):
    r = raw[split].loc[rid]
    flow = {c: clean(r[c]) for c in FLOW_COLS}
    state = {'network_flow': flow, 'dataset': 'UNSW-NB15 flow record (features only)'}
    if variant == 'full':
        s = sig[split].loc[rid]
        state['detector'] = {'random_forest_attack_probability': round(float(s.p_attack), 3),
                             'tree_disagreement_std': round(float(s.tree_std), 3)}
        state['explanation'] = {
            'top_shap_features_toward_attack_if_positive': parse_top(s.top_feats),
            'xai_q': {'stability_ESS': round(float(s.ESS), 3), 'concentration_SI': round(float(s.SI), 3),
                      'analyst_alignment_AR': round(float(s.AR), 3)},
        }
    return state


example = build_state('test', sig['test'].index[0], 'full')
print(json.dumps(example, indent=1)[:1800])

val alerts: 2853 | raw/label alignment OK
test alerts: 3622 | raw/label alignment OK
{
 "network_flow": {
  "proto": "udp",
  "service": "-",
  "state": "INT",
  "dur": 1e-05,
  "spkts": 2,
  "dpkts": 0,
  "sbytes": 2126,
  "dbytes": 0,
  "rate": 100000.0,
  "sttl": 254,
  "dttl": 0,
  "sload": 850400000.0,
  "dload": 0.0,
  "sloss": 0,
  "dloss": 0,
  "sinpkt": 0.01,
  "dinpkt": 0.0,
  "sjit": 0.0,
  "djit": 0.0,
  "tcprtt": 0.0,
  "smean": 1063,
  "dmean": 0,
  "ct_srv_src": 3,
  "ct_dst_ltm": 2,
  "ct_src_dport_ltm": 2,
  "ct_dst_sport_ltm": 1,
  "ct_state_ttl": 2,
  "ct_flw_http_mthd": 0,
  "is_ftp_login": 0
 },
 "dataset": "UNSW-NB15 flow record (features only)",
 "detector": {
  "random_forest_attack_probability": 0.741,
  "tree_disagreement_std": 0.087
 },
 "explanation": {
  "top_shap_features_toward_attack_if_positive": [
   {
    "feature": "sttl",
    "shap_toward_attack": 0.087
   },
   {
    "feature": "ct_state_ttl",
    "shap_toward_attack": 0.059
   },
   {
    "feature

## 2. Questions and API call (with retry, cache and mock)

In [15]:
def make_questions():
    from typesafe_sdk import Choice, Noul, Score
    return {
        'is_malicious': Noul(instructions='This network flow is part of a cyber attack or intrusion (not benign traffic).'),
        'action': Choice(
            instructions='What should the security system do with this flow alert?',
            criteria={'block': 'Clearly malicious: block automatically',
                      'allow': 'Clearly benign: allow automatically',
                      'escalate': 'Ambiguous or unusual: send to a human security analyst'}),
        'explanation_supports': Score(
            instructions='If an explanation is given, how well do the listed top features and stability metrics support the verdict '
                         'with behaviour a network analyst would accept? If no explanation is given, judge the flow evidence only.',
            criteria=['Unsupported or based on non-behavioural artefacts', 'Partly supported', 'Well supported by traffic behaviour']),
    }


_tls = threading.local()
_lock = threading.Lock()


def _client():
    if not hasattr(_tls, 'c'):
        from typesafe_sdk import TypeSafeClient
        _tls.c = TypeSafeClient(model=JEV_MODEL)   # reads TYPESAFE_API_KEY
    return _tls.c


def mock_answer(p_hint, rng):
    # Pipeline test only: a noisy function of the RF probability. Never use for results.
    z = np.log(np.clip(p_hint, 1e-3, 1 - 1e-3) / (1 - np.clip(p_hint, 1e-3, 1 - 1e-3))) + rng.normal(0, 1.0)
    noul = 1 / (1 + np.exp(-z))
    act = 'block' if noul > 0.8 else ('allow' if noul < 0.15 else 'escalate')
    conf = float(abs(noul - 0.5) * 2)
    return {'noul': float(noul), 'action': act, 'action_conf': conf, 'action_probs': {act: conf}, 'expl_score': float(rng.uniform(0, 2)),
            'model': 'mock'}


def call_jev(state, hint_p):
    if USE_MOCK:
        return mock_answer(hint_p, np.random.default_rng(abs(hash(json.dumps(state, sort_keys=True))) % 2**32))
    last = None
    for k in range(MAX_RETRIES):
        try:
            r = _client().system_one(model=JEV_MODEL, state=state, questions=make_questions())
            a = r.answers
            return {'noul': float(a['is_malicious'].noul), 'action': str(a['action'].choice), 'action_conf': float(a['action'].confidence),
                    'action_probs': {k2: float(v) for k2, v in a['action'].probabilities.items()},
                    'expl_score': float(a['explanation_supports'].score), 'model': getattr(r, 'model', None),
                    'usage': getattr(getattr(r, 'usage', None), '__dict__', None) or getattr(r, 'usage', None)}
        except Exception as e:   # network / rate limit / server errors: back off and retry
            if type(e).__name__ in ('TypeSafeAuthenticationError', 'TypeSafePermissionDeniedError') or ' 401 ' in str(e) or ' 403 ' in str(e):
                k_ = os.getenv('TYPESAFE_API_KEY', '')
                raise RuntimeError(f'Auth rejected (not retried). key length={len(k_)}, startswith={k_[:3]!r}, has_space_or_quote={any(c in k_ for c in " \'\"\\n\\r")}. {e}') from None
            last = e
            time.sleep(2 ** k)
    raise RuntimeError(f'Jev call failed after {MAX_RETRIES} tries: {type(last).__name__}: {last}')


def load_cache():
    done = {}
    if CACHE.exists():
        for line in CACHE.read_text(encoding='utf-8').splitlines():
            d = json.loads(line)
            done[(d['split'], int(d['row_id']), d['variant'])] = d
    return done


def run_jobs(jobs):
    done = load_cache()
    todo = [j for j in jobs if j not in done]
    print(f'{len(jobs)} requested | {len(jobs) - len(todo)} cached | {len(todo)} to send')

    def work(j):
        split, rid, variant = j
        res = call_jev(build_state(split, rid, variant), float(sig[split].loc[rid, 'p_attack']))
        rec = {'split': split, 'row_id': int(rid), 'variant': variant, **res}
        with _lock, open(CACHE, 'a', encoding='utf-8') as f:
            f.write(json.dumps(rec, default=str) + '\n')
        return rec

    t0 = time.time()
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        for i, _ in enumerate(ex.map(work, todo), 1):
            if i % 200 == 0:
                print(f'  {i}/{len(todo)} done, {time.time() - t0:.0f}s')
    print(f'finished in {time.time() - t0:.0f}s')

## 3. Send the requests
Dry run first (about `DRY_PER_CAT` alerts per category per split per variant); then set `DRY_RUN = False` and run again - cached answers are reused.

In [18]:
def select_ids(split):
    s = sig[split]
    if DRY_RUN:
        return list(s.groupby('attack_cat', group_keys=False).apply(lambda g: g.sample(min(len(g), DRY_PER_CAT), random_state=SEED)).index)
    return list(s.index)


jobs = [(sp, rid, v) for sp in ('val', 'test') for rid in select_ids(sp) for v in VARIANTS]
run_jobs(jobs)

120 requested | 0 cached | 120 to send
finished in 14s


In [19]:
cache = load_cache()
J = pd.DataFrame(list(cache.values()))
print('cached answers:', len(J), '| API model(s):', J['model'].dropna().unique().tolist())
J = J[J.apply(lambda r: (r['split'], r['row_id'], r['variant']) in set(jobs), axis=1)]


def with_truth(df):
    d = df.copy()
    d['label'] = [sig[s].loc[r, 'label'] for s, r in zip(d.split, d.row_id)]
    d['attack_cat'] = [sig[s].loc[r, 'attack_cat'] for s, r in zip(d.split, d.row_id)]
    d['rf_p'] = [sig[s].loc[r, 'p_attack'] for s, r in zip(d.split, d.row_id)]
    return d


Jt = with_truth(J)
rows = {}
for (sp, v), g in Jt.groupby(['split', 'variant']):
    rows[f'{sp}/{v}'] = {'n': len(g), 'AUROC noul': roc_auc_score(g.label, g.noul) if g.label.nunique() > 1 else np.nan,
                         'AUROC RF': roc_auc_score(g.label, g.rf_p) if g.label.nunique() > 1 else np.nan,
                         'noul | attack': g.loc[g.label == 1, 'noul'].mean(), 'noul | normal': g.loc[g.label == 0, 'noul'].mean(),
                         'block %': 100 * (g.action == 'block').mean(), 'allow %': 100 * (g.action == 'allow').mean(),
                         'escalate %': 100 * (g.action == 'escalate').mean(), 'mean conf': g.action_conf.mean()}
pd.DataFrame(rows).T.round(3)

cached answers: 120 | API model(s): ['jev-1.14.0']


,n,AUROC noul,AUROC RF,noul | attack,noul | normal,block %,allow %,escalate %,mean conf
test/flow_only,30.0,0.642,1.0,0.418,0.370,0.0,3.333,96.667,0.565
test/full,30.0,1.000,1.0,0.705,0.270,0.0,6.667,93.333,0.560
val/flow_only,30.0,0.611,1.0,0.343,0.277,0.0,3.333,96.667,0.548
val/full,30.0,1.000,1.0,0.694,0.090,0.0,10.000,90.000,0.562


## 4. Full evaluation (runs only when every sampled alert has an answer)
Same protocol as notebook 04: thresholds / fusion model are chosen on **validation** only, test is evaluated once, weighted to the real traffic mix, with a stratified paired bootstrap.

In [26]:
need = {(sp, rid, v) for sp in ('val', 'test') for rid in sig[sp].index for v in VARIANTS}
have = set(cache.keys())
COMPLETE = need <= have
print('full evaluation possible:', COMPLETE, f'({len(need & have)}/{len(need)} answers cached)')
COMPLETE = True

full evaluation possible: False (120/12950 answers cached)


In [27]:
GRID = np.r_[np.linspace(0, 1, 401), 1.0001]


def choose_thresholds(p, y, w, c_fn=C_FN, c_fp=C_FP, c_esc=C_ESC):
    o = np.argsort(p)
    ps, ws, ys = p[o], w[o], y[o]
    cum_fn = np.r_[0, np.cumsum(ws * ys * c_fn)]
    cum_fp = np.r_[0, np.cumsum(ws * (1 - ys) * c_fp)]
    cum_w = np.r_[0, np.cumsum(ws)]
    i = np.searchsorted(ps, GRID, 'left')
    cost = cum_fn[i][:, None] + (cum_fp[-1] - cum_fp[i])[None, :] + c_esc * (cum_w[i][None, :] - cum_w[i][:, None])
    cost = np.where(GRID[None, :] >= GRID[:, None], cost, np.inf) / cum_w[-1]
    a, b = np.unravel_index(np.argmin(cost), cost.shape)
    return GRID[a], GRID[b]


def typed(p, lo, hi):
    return np.where(p < lo, 'ALLOW', np.where(p >= hi, 'BLOCK', 'ESCALATE'))


def row_cost(d, y):
    return np.where(d == 'ALLOW', y * C_FN, np.where(d == 'BLOCK', (1 - y) * C_FP, C_ESC))


def summarize(d, y, w):
    auto = d != 'ESCALATE'
    fn, fp = (d == 'ALLOW') & (y == 1), (d == 'BLOCK') & (y == 0)
    return {'cost/alert': np.average(row_cost(d, y), weights=w), 'escalate %': 100 * w[~auto].sum() / w.sum(),
            'auto error %': 100 * w[fn | fp].sum() / max(w[auto].sum(), 1e-9),
            'missed attacks %': 100 * w[fn].sum() / w[y == 1].sum(),
            'false blocks % of normal': 100 * w[fp].sum() / w[y == 0].sum(),
            'attack share in queue %': 100 * w[~auto & (y == 1)].sum() / max(w[~auto].sum(), 1e-9)}


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


if COMPLETE:
    Jf = with_truth(pd.DataFrame(list(cache.values())))
    yv, yt = sig['val'].label.values, sig['test'].label.values
    wv, wt = sig['val'].sample_weight.values, sig['test'].sample_weight.values
    cats = sig['test'].attack_cat.values

    def table(split, variant):
        g = Jf[(Jf.split == split) & (Jf.variant == variant)].set_index('row_id').loc[sig[split].index]
        return g

    D, P = {}, {}
    sur = pd.read_parquet(OUT_DIR / 'jev_test_decisions.parquet')
    D['RF alone @0.5'] = np.where(sig['test'].p_attack.values >= 0.5, 'BLOCK', 'ALLOW')
    D['Surrogate JEV (nb04)'] = sur.loc[sig['test'].index, 'decision'].values

    for v in VARIANTS:
        gv, gt = table('val', v), table('test', v)
        # (a) Jev's own typed decision; low confidence -> escalate (docs: confidence floor)
        d = np.where(gt.action_conf.values < CONF_FLOOR, 'ESCALATE', gt.action.str.upper().values)
        D[f'Jev [{v}] typed action + conf floor'] = d
        # (b) Jev's Noul probability, thresholds chosen on validation by cost
        lo, hi = choose_thresholds(gv.noul.values, yv, wv)
        D[f'Jev [{v}] noul + cost thresholds'] = typed(gt.noul.values, lo, hi)
        print(f'{v}: noul thresholds t_lo={lo:.3f} t_hi={hi:.3f}')

    # (c) fusion of Jev outputs with the RF + XAI-Q signals (logistic regression, OOF thresholds on validation)
    def fuse_X(split, v):
        g, s = table(split, v), sig[split]
        return pd.DataFrame({'logit_p': logit(s.p_attack.values), 'ESS': s.ESS.values, 'SI': s.SI.values, 'tree_std': s.tree_std.values,
                             'jev_logit': logit(g.noul.values), 'jev_conf': g.action_conf.values, 'jev_expl': g.expl_score.values})

    for v in VARIANTS:
        Xv, Xt = fuse_X('val', v), fuse_X('test', v)
        oof = np.zeros(len(Xv))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xv, sig['val'].attack_cat):
            m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
            m.fit(Xv.iloc[tr], yv[tr], logisticregression__sample_weight=wv[tr] / wv[tr].mean())
            oof[te] = m.predict_proba(Xv.iloc[te])[:, 1]
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        m.fit(Xv, yv, logisticregression__sample_weight=wv / wv.mean())
        P[v] = m.predict_proba(Xt)[:, 1]
        lo, hi = choose_thresholds(oof, yv, wv)
        D[f'Fusion RF+XAI-Q+Jev [{v}]'] = typed(P[v], lo, hi)

    res = pd.DataFrame({k: summarize(d, yt, wt) for k, d in D.items()}).T
    res['cost reduction vs RF@0.5 %'] = 100 * (1 - res['cost/alert'] / res.loc['RF alone @0.5', 'cost/alert'])
    display(res.round(3))
else:
    print('skipped: run the full set first (DRY_RUN = False)')

KeyError: '[3, 203, 220, 837, 1321, 1488, 1521, 1774, 1833, 1869, 2117, 2433, 2609, 2748, 3460, 3481, 3498, 3529, 3612, 3666, 4106, 4355, 4400, 4430, 4683, 4741, 5116, 5236, 5355, 5440, 5742, 5801, 5895, 6281, 6363, 6499, 6564, 6572, 6596, 6634, 7078, 7166, 7368, 7373, 7406, 7598, 7778, 8095, 8137, 8267, 8376, 8567, 8634, 8678, 8691, 8951, 8973, 8982, 9013, 9137, 9357, 9427, 9599, 9693, 9744, 9855, 9862, 10104, 10341, 10487, 10517, 10769, 10974, 11136, 11196, 11329, 11388, 11544, 11568, 11736, 11831, 11841, 11864, 11882, 11935, 12095, 12349, 12356, 12365, 12459, 12569, 12669, 12768, 12817, 12965, 13118, 13169, 13218, 13369, 13603, 13604, 13608, 13687, 13758, 13771, 13806, 13857, 13893, 14012, 14078, 14374, 14750, 15089, 15256, 15455, 15586, 15638, 15747, 15778, 15950, 16228, 16378, 16616, 16714, 16843, 17127, 17218, 17267, 17270, 17391, 17681, 17692, 17754, 17811, 17955, 18077, 18262, 18349, 18559, 18564, 18615, 18951, 19035, 19219, 19314, 19558, 19670, 20048, 20162, 21026, 21150, 2121

In [ ]:
if COMPLETE:
    rng = np.random.default_rng(SEED)
    groups = [np.where(cats == c)[0] for c in np.unique(cats)]
    idx = np.concatenate([g[rng.integers(0, len(g), (N_BOOT, len(g)))] for g in groups], axis=1)
    RC = {k: row_cost(d, yt) for k, d in D.items()}
    boot = {k: (wt[idx] * c[idx]).sum(1) / wt[idx].sum(1) for k, c in RC.items()}
    ref = 'Surrogate JEV (nb04)'
    out = []
    for k in D:
        diff = boot[k] - boot[ref]
        out.append({'method': k, 'cost/alert': RC[k] @ wt / wt.sum(), 'CI low': np.percentile(boot[k], 2.5), 'CI high': np.percentile(boot[k], 97.5),
                    f'diff vs {ref}': diff.mean(), 'diff CI low': np.percentile(diff, 2.5), 'diff CI high': np.percentile(diff, 97.5),
                    'P(better)': (diff < 0).mean()})
    display(pd.DataFrame(out).set_index('method').round(3))

## 5. Does Jev's output add information beyond the RF? Per-category behaviour and calibration of the Noul

In [ ]:
if COMPLETE:
    def ece(p, y, w, bins=10):
        b = np.minimum((p * bins).astype(int), bins - 1)
        return sum(w[b == k].sum() / w.sum() * abs(np.average(y[b == k], weights=w[b == k]) - np.average(p[b == k], weights=w[b == k]))
                   for k in range(bins) if (b == k).any())

    rows = {'RF raw': (sig['test'].p_attack.values, None)}
    for v in VARIANTS:
        rows[f'Jev noul [{v}]'] = (table('test', v).noul.values, None)
        rows[f'Fusion [{v}]'] = (P[v], None)
    display(pd.DataFrame({k: {'AUROC': roc_auc_score(yt, p, sample_weight=wt), 'ECE': ece(p, yt, wt),
                              'Brier': np.average((p - yt) ** 2, weights=wt)} for k, (p, _) in rows.items()}).T.round(4))

    key = f'Fusion RF+XAI-Q+Jev [{VARIANTS[-1]}]'
    dd = D[key]
    pc = {}
    for c in np.unique(cats):
        m = cats == c
        s = summarize(dd[m], yt[m], wt[m])
        pc[c] = {'est. alerts': wt[m].sum(), 'escalate %': s['escalate %'], 'auto error %': s['auto error %']}
    display(pd.DataFrame(pc).T.round(2))

## 6. Save

In [ ]:
if COMPLETE:
    res.round(4).to_csv(OUT_DIR / ('jev_api_results_test_MOCK.csv' if USE_MOCK else 'jev_api_results_test.csv'))
    pd.DataFrame(D, index=sig['test'].index).to_parquet(OUT_DIR / ('jev_api_decisions_test_MOCK.parquet' if USE_MOCK else 'jev_api_decisions_test.parquet'))
    print('saved')
else:
    print('nothing saved (incomplete run)')